## Plot fraction of individuals with low neutralization titers by strain

In [1]:
# Import packages
import os
import altair as alt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gmean
from IPython.display import IFrame


# Ignore error message from Altair about large dataframes
_ = alt.data_transformers.disable_max_rows()

# Basic color palette
color_palette = [
    '#345995', #blue
    '#03cea4', #teal
    '#ca1551', #red
    '#eac435', #yellow
               ]

In [2]:
# Define inputs
resultsdir = '../results'
os.makedirs(resultsdir, exist_ok = True)

# Define titers
titers = pd.read_csv(
    '../../../results/aggregated_titers/titers_EllebedyVaccineCohort.csv'
)

titers = titers.replace({'mRNA_': 'mRNA-1010_'})
titers['serum_new'] = titers['serum'].str.replace('mRNA_', 'mRNA-1010_', regex=False)
titers = titers.drop('serum', axis=1).rename(columns={'serum_new': 'serum'})


titers = titers.assign(
    vaccine = lambda x: x['serum'].str.split('_').str[0],
    participant = lambda x: x['serum'].str.split('_').str[1],
    timepoint = lambda x: x['serum'].str.split('_').str[2],
    subtype = lambda x: x['virus'].str.split('_').str[-1]
)


# Calculate number of unique participants per vaccine
n_participants = titers.groupby('vaccine')['participant'].nunique()

# Create a mapping dictionary
vaccine_pretty = {v: f"{v} (n={n_participants[v]})" for v in n_participants.index}

# Map to a new pretty column
titers['vaccine_pretty'] = titers['vaccine'].map(vaccine_pretty)
titers['vaccine'] = titers['vaccine'].map(vaccine_pretty)

titers.serum.unique()

array(['Fluarix_397-001_d0', 'Fluarix_397-001_d181',
       'Fluarix_397-001_d29', 'Fluarix_397-002_d0',
       'Fluarix_397-002_d181', 'Fluarix_397-002_d29',
       'Fluarix_397-005_d0', 'Fluarix_397-005_d181',
       'Fluarix_397-005_d29', 'Fluarix_397-007_d0',
       'Fluarix_397-007_d121', 'Fluarix_397-007_d29',
       'Fluarix_397-009_d0', 'Fluarix_397-009_d121',
       'Fluarix_397-009_d29', 'Fluarix_397-011_d0',
       'Fluarix_397-011_d121', 'Fluarix_397-011_d29',
       'Fluarix_397-016_d0', 'Fluarix_397-016_d181',
       'Fluarix_397-016_d29', 'Fluarix_397-018_d0',
       'Fluarix_397-018_d121', 'Fluarix_397-018_d29',
       'Fluarix_397-019_d0', 'Fluarix_397-019_d181',
       'Fluarix_397-019_d29', 'Fluarix_397-020_d0',
       'Fluarix_397-020_d181', 'Fluarix_397-020_d29',
       'Fluarix_397-022_d0', 'Fluarix_397-022_d181',
       'Fluarix_397-022_d29', 'Fluarix_397-024_d0',
       'Fluarix_397-024_d181', 'Fluarix_397-024_d29',
       'Fluarix_397-026_d0', 'Fluarix_397-026_

In [3]:
# Define virus order
viral_plot_order = pd.read_csv('../../../data/H3_H1_ellebedy_library_strain_order.csv')
virus_order = [v for v in viral_plot_order.strain]

## Plot neutralization titers

In [4]:
def plot_titers_vaccination_cohorts(data, sort_order, _range = [30, 20000], title=None):
    # Make plot with all individuals and median dots
    color_scheme = alt.Color('timepoint', sort=['d0', 'd29']).scale(range=color_palette)
    titer_range = _range
    titleFontSize=18
    labelFontSize=18
    lineOpacity = 0.2
    lineSize = 2.8
    markerOpacity = 0.8
    markerSize = 160

    band = (alt.Chart(data)
            .mark_errorband(extent='iqr', opacity=0.4)
            .encode(alt.X('virus', axis = alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize,
                                          title = None,labelLimit = 1000, labelAlign = 'right'),             
                          sort = virus_order),
                    alt.Y('titer', 
                          scale =alt.Scale(type='log',domain=_range, nice=False), 
                          axis=alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize, title="neutralization titer")),
                color = color_scheme,)
           ) 
    
    points = (alt.Chart(data)
              .mark_point(size = markerSize, stroke = 'black', strokeWidth = 2.2, filled=True,  opacity=markerOpacity)
              .encode(alt.X('virus', sort = virus_order),
                      alt.Y('median(titer)'),
                      color = color_scheme,))
        
    layered = (alt.layer(band, points)
               .facet(row = alt.Row('vaccine:N', sort=sort_order),
                      config = alt.Config(legend = alt.LegendConfig(titleFontSize=titleFontSize, labelFontSize = labelFontSize,
                                                    strokeColor='gray',padding=10,cornerRadius=10,
                                                    labelLimit = 1000 # Let legend labels be as long as they want
                                                     )))
               .properties(title=title)
               .configure_header(title=None,
                                  labelFontSize=labelFontSize,labelFontWeight='bold',
                                  labelOrient='right', 
                                )
               .configure_title(align='center', anchor='middle', fontSize=titleFontSize, fontWeight='bold')
           .configure_legend(symbolSize=markerSize, symbolOpacity=markerOpacity, symbolStrokeWidth=2.2, symbolStrokeColor='black', 
                             titleFontSize=titleFontSize, labelFontSize = labelFontSize,
                            strokeColor='gray',padding=10,cornerRadius=10,
                            labelLimit = 1000 # Let legend labels be as long as they want
                            )
    )

    return layered

In [ ]:
for subtype in titers.subtype.unique():
    data = titers[titers['subtype'].isin([subtype])]

    plot = plot_titers_vaccination_cohorts(data, sort_order = ['Fluarix', 'mRNA-1010'], 
                                           _range=[30, 18000], title = f'{subtype} 2023-2024 circulating strains and recent vaccine strains')
    # Save final plot
    outfile = os.path.join(resultsdir, f'{subtype}_pre-post_titers.pdf')
    plot.save(outfile, dpi = 600)


In [ ]:
IFrame(os.path.join(resultsdir, f'H3N2_pre-post_titers.pdf'), width=700, height=500)

In [ ]:
IFrame(os.path.join(resultsdir, f'H1N1_pre-post_titers.pdf'), width=700, height=500)

# Plot log fold-change titers for each group from day 0 to day 29 and day 0 to day 121/181

In [ ]:
# Pivot the table to have timepoints as columns
pivot_titers = titers.pivot_table(
    index=['participant', 'vaccine', 'virus', 'subtype', ],
    columns='timepoint',
    values='titer'
)

# Calculate fold-changes
pivot_titers['fold_change_d29'] = pivot_titers['d29'] / pivot_titers['d0']
if 'd121' in pivot_titers.columns:
    pivot_titers['fold_change_d121'] = pivot_titers['d121'] / pivot_titers['d0']
pivot_titers['fold_change_d181'] = pivot_titers['d181'] / pivot_titers['d0']

# There's inconsistency between the last date being d121 or d181, so making a column that has both
if 'd121' in pivot_titers.columns:
    pivot_titers['fold_change_d121_or_d181'] = pivot_titers.get('d121', pd.NA).combine_first(pivot_titers.get('d181', pd.NA)) / pivot_titers['d0']

# Compute log2 fold changes
pivot_titers['log2_fc_d29'] = np.log2(pivot_titers['fold_change_d29'])
pivot_titers['log2_fc_d181'] = np.log2(pivot_titers['fold_change_d181'])
if 'd121' in pivot_titers.columns:
    pivot_titers['log2_fc_d121'] = np.log2(pivot_titers['fold_change_d121'])
    pivot_titers['log2_fc_d121_or_d181'] = np.log2(pivot_titers['fold_change_d121_or_d181'])

pivot_titers = pivot_titers.reset_index()
pivot_titers


In [ ]:
def plot_FC_vaccination_cohorts(data, y_axis, sort_order, _range = [0, 1], title=None):
    # Make plot with all individuals and median dots
    color_scheme = alt.Color('vaccine', sort=['mRNA-1010', 'Fluarix']).scale(range=color_palette)
    titer_range = _range
    titleFontSize=18
    labelFontSize=18
    lineOpacity = 0.2
    lineSize = 2.8
    markerOpacity = 0.8
    markerSize = 160

    band = (alt.Chart(data)
            .mark_errorband(extent='iqr', opacity=0.4)
            .encode(alt.X('virus', axis = alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize,
                                          title = None,labelLimit = 1000, labelAlign = 'right'),             
                          sort = virus_order),
                    alt.Y(f'{y_axis}', 
                          scale =alt.Scale(domain=_range, nice=False), 
                          axis=alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize, title=["log FC titer", "from pre-vaccination"])),
                color = color_scheme,)
           ) 
    
    points = (alt.Chart(data)
              .mark_point(size = markerSize, stroke = 'black', strokeWidth = 2.2, filled=True,  opacity=markerOpacity)
              .encode(alt.X('virus', sort = virus_order),
                      alt.Y(f'median({y_axis})'),
                      color = color_scheme,))
        
    layered = (alt.layer(band, points)
               .properties(title=title)
               .configure_header(title=None,
                                  labelFontSize=labelFontSize,labelFontWeight='bold',
                                  labelOrient='right', 
                                )
               .configure_title(align='center', anchor='middle', fontSize=titleFontSize, fontWeight='bold')
           .configure_legend(symbolSize=markerSize, symbolOpacity=markerOpacity, symbolStrokeWidth=2.2, symbolStrokeColor='black', 
                             titleFontSize=titleFontSize, labelFontSize = labelFontSize,
                            strokeColor='gray',padding=10,cornerRadius=10,
                            labelLimit = 1000 # Let legend labels be as long as they want
                            )
    )

    return layered

## Log FC in neutralization titers at day 29

In [ ]:
for subtype in titers.subtype.unique():
    data = pivot_titers[pivot_titers['subtype'].isin([subtype])]

    plot = plot_FC_vaccination_cohorts(data, 
                                           y_axis = 'log2_fc_d29',
                                           sort_order = ['mRNA-1010', 'Fluarix'], 
                                           _range=[-0.6, 6], title = f'Log FC day 29 titers to {subtype} 2023-2024 circulating strains and recent vaccine strains')
    # Save final plot
    outfile = os.path.join(resultsdir, f'{subtype}_day29_pre-post_fold-change.pdf')
    plot.save(outfile, dpi = 600)


In [ ]:
IFrame(os.path.join(resultsdir, 'H3N2_day29_pre-post_fold-change.pdf'), width=700, height=500)

In [ ]:
IFrame(os.path.join(resultsdir, 'H1N1_day29_pre-post_fold-change.pdf'), width=700, height=500)

## Log FC in neutralization titers at day 121 or 181

In [ ]:
if 'd121' in pivot_titers.columns:
    y_axis = 'log2_fc_d121_or_d181'
else:
    y_axis = 'log2_fc_d181'

for subtype in titers.subtype.unique():
    data = pivot_titers[pivot_titers['subtype'].isin([subtype])]

    plot = plot_FC_vaccination_cohorts(data, 
                                           y_axis = y_axis,
                                           sort_order = ['mRNA-1010', 'Fluarix', ], 
                                           _range=[-1.5, 4.5], title = f'Log FC day 121/181 titers to {subtype} 2023-2024 circulating strains and recent vaccine strains')
    # Save final plot
    outfile = os.path.join(resultsdir, f'{subtype}_day121-181_pre-post_fold-change.pdf')
    plot.save(outfile, dpi = 600)


In [ ]:
IFrame(os.path.join(resultsdir, 'H3N2_day121-181_pre-post_fold-change.pdf'), width=700, height=500)

In [ ]:
IFrame(os.path.join(resultsdir, 'H1N1_day121-181_pre-post_fold-change.pdf'), width=700, height=500)